<a href="https://colab.research.google.com/github/ABHILASHTIWARI861/ABHILASHTIWARI861/blob/main/ADVANCED_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import time

print("GPU:", tf.config.list_physical_devices('GPU'))

import tensorflow as tf
from tensorflow.keras import layers, models
import time

print("GPU:", tf.config.list_physical_devices('GPU'))


(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0

batch_size = 64

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)) \
    .shuffle(50000).batch(batch_size)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)) \
    .batch(batch_size)

def build_standard_cnn():
    model = models.Sequential([
        layers.Conv2D(32, 3, padding='same', activation='relu', input_shape=(32,32,3)),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10)
    ])
    return model

def build_efficient_cnn():  # separable
    model = models.Sequential([
        layers.SeparableConv2D(32, 3, padding='same', activation='relu', input_shape=(32,32,3)),
        layers.MaxPooling2D(),

        layers.SeparableConv2D(64, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10)
    ])
    return model


def build_dilated_cnn():
    model = models.Sequential([
        layers.Conv2D(32, 3, padding='same', dilation_rate=2, activation='relu', input_shape=(32,32,3)),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding='same', dilation_rate=2, activation='relu'),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10)
    ])
    return model


def build_no_pool_cnn():
    model = models.Sequential([
        layers.Conv2D(32, 3, strides=2, padding='same', activation='relu', input_shape=(32,32,3)),
        layers.Conv2D(64, 3, strides=2, padding='same', activation='relu'),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10)
    ])
    return model

def build_random_cnn():
    inputs = layers.Input(shape=(32,32,3))

    x = layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)

    # Freeze convolution layer
    conv_layer = models.Model(inputs, x)
    conv_layer.trainable = False

    x = conv_layer(inputs)
    x = layers.MaxPooling2D()(x)

    x = layers.Flatten()(x)
    outputs = layers.Dense(10)(x)

    return models.Model(inputs, outputs)

from tensorflow.keras import layers, models

def build_avg_pool_cnn():
    model = models.Sequential([
        layers.Conv2D(32, 3, padding='same', activation='relu', input_shape=(32,32,3)),
        layers.AveragePooling2D(pool_size=2),

        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.AveragePooling2D(pool_size=2),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10)
    ])
    return model

def train_and_evaluate(model, name, epochs=5):
    print(f"\n==== {name} ====")

    model.compile(
        optimizer='adam',
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )

    start = time.time()

    history = model.fit(train_ds, epochs=epochs, validation_data=test_ds, verbose=1)

    end = time.time()
    print(f"Training Time: {end - start:.2f} sec")

    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    print(f"Test Accuracy: {test_acc*100:.2f}%")

    return history

models_dict = {
    "Standard CNN": build_standard_cnn(),
    "Efficient CNN": build_efficient_cnn(),
    "Dilated CNN": build_dilated_cnn(),
    "No Pooling CNN": build_no_pool_cnn(),
    "Random CNN": build_random_cnn(),
}

histories = {}

for name, model in models_dict.items():
    histories[name] = train_and_evaluate(model, name, epochs=5)


histories['Avg Pooling CNN'] = train_and_evaluate(build_avg_pool_cnn(), 'Avg Pooling CNN', epochs=5)
import matplotlib.pyplot as plt

for name, hist in histories.items():
    plt.plot(hist.history['val_accuracy'], label=name)

plt.title("Model Comparison")
plt.xlabel("Epochs")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.show()



GPU: []
GPU: []
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_separable_conv.py:104: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(



==== Standard CNN ====
Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 89s 110ms/step - accuracy: 0.4924 - loss: 1.4225 - val_accuracy: 0.5775 - val_loss: 1.2009
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 85s 108ms/step - accuracy: 0.6334 - loss: 1.0480 - val_accuracy: 0.6454 - val_loss: 1.0006
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 85s 109ms/step - accuracy: 0.6880 - loss: 0.8970 - val_accuracy: 0.6892 - val_loss: 0.9008
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 83s 106ms/step - accuracy: 0.7213 - loss: 0.8004 - val_accuracy: 0.7056 - val_loss: 0.8517
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 83s 106ms/step - accuracy: 0.7473 - loss: 0.7279 - val_accuracy: 0.7012 - val_loss: 0.8727
Training Time: 483.83 sec
Test Accuracy: 70.12%

==== Efficient CNN ====
Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 67s 84ms/step - accuracy: 0.4267 - loss: 1.5962 - val_accuracy: 0.5217 - val_loss: 1.3462
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 59s 76ms/step - accuracy: 0.5451 - loss: 1.2809 - val_accuracy: 0.5665 - val_loss: 1.